# Practice: maximum likelihood, priors and ridge
*AI from scratch, probability 3* · lesson: https://ihor.sh/ai/p3-mle-and-map.html

Fit distributions by maximum likelihood (by formula, by SciPy, by gradient descent in PyTorch), watch the central
limit theorem, update a Beta prior, and confirm that ridge regression is MAP with a Gaussian prior.
Fill in each `...` under `# YOUR CODE HERE`, then run the check under it (✓ = right, ✗ = not yet).
On Kaggle, turn on *Internet* in the notebook settings for part 1's data.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['Take natural logarithm of outage durations: logs = np.log(hours).', 'Compute MLE parameters mu_hat = logs.mean() and sigma_hat = logs.std().'], 2: ['Initialize trainable parameters mu and s = ln(sigma) as zero tensors with requires_grad=True.', 'Set up Adam optimizer: torch.optim.Adam([mu, s], lr=0.05).', 'Run 1000 steps: compute NLL = sum((x - mu)**2 / (2 * exp(2*s))) + n * s; opt.zero_grad(); nll.backward(); opt.step().', 'Extract scalar values: mu_gd = mu.item(), sigma_gd = torch.exp(s).item().'], 3: ['Generate 100,000 sums of 30 exponential draws: rng.exponential(4, (100_000, 30)).sum(axis=1).', 'Compute skewness using stats.skew(sums).'], 4: ['Update Beta conjugate parameters: a_post = a_prior + heads, b_post = b_prior + tails.', 'Compute MAP mode of Beta: p_map = (a_post - 1) / (a_post + b_post - 2).', 'Compute 95% credible interval: stats.beta.ppf([0.025, 0.975], a_post, b_post).'], 5: ['Solve the MAP formula: np.linalg.solve(X.T @ X + 50 * np.eye(X.shape[1]), X.T @ y).', 'Fit Ridge(alpha=50, fit_intercept=False).fit(X, y) and extract .coef_.']}
_ANSWERS = {1: 'bG9ncyA9IG5wLmxvZyhob3VycykKbXVfaGF0LCBzaWdtYV9oYXQgPSBsb2dzLm1lYW4oKSwgbG9ncy5zdGQoKQ==', 2: 'bXUgPSB0b3JjaC56ZXJvcygoKSwgcmVxdWlyZXNfZ3JhZD1UcnVlLCBkdHlwZT10b3JjaC5mbG9hdDY0KTsgcyA9IHRvcmNoLnplcm9zKCgpLCByZXF1aXJlc19ncmFkPVRydWUsIGR0eXBlPXRvcmNoLmZsb2F0NjQpCm9wdCA9IHRvcmNoLm9wdGltLkFkYW0oW211LCBzXSwgbHI9MC4wNSkKZm9yIF8gaW4gcmFuZ2UoMTAwMCk6CiAgICBvcHQuemVyb19ncmFkKCkKICAgIG5sbCA9ICgoeCAtIG11KSAqKiAyIC8gKDIgKiB0b3JjaC5leHAoMiAqIHMpKSkuc3VtKCkgKyBsZW4oeCkgKiBzCiAgICBubGwuYmFja3dhcmQoKTsgb3B0LnN0ZXAoKQptdV9nZCwgc2lnbWFfZ2QgPSBtdS5pdGVtKCksIHRvcmNoLmV4cChzKS5pdGVtKCk=', 3: 'c3VtcyA9IHJuZy5leHBvbmVudGlhbCg0LCAoMTAwXzAwMCwgMzApKS5zdW0oYXhpcz0xKQpza2V3X3N1bSA9IHN0YXRzLnNrZXcoc3Vtcyk=', 4: 'YV9wb3N0LCBiX3Bvc3QgPSAyICsgNywgMiArIDMKcF9tYXAgPSAoYV9wb3N0IC0gMSkgLyAoYV9wb3N0ICsgYl9wb3N0IC0gMikKaW50ZXJ2YWwgPSBzdGF0cy5iZXRhLnBwZihbMC4wMjUsIDAuOTc1XSwgYV9wb3N0LCBiX3Bvc3Qp', 5: 'd19tYXAgPSBucC5saW5hbGcuc29sdmUoWC5UIEAgWCArIDUwICogbnAuZXllKFguc2hhcGVbMV0pLCBYLlQgQCB5KQp3X3JpZGdlID0gUmlkZ2UoYWxwaGE9NTAsIGZpdF9pbnRlcmNlcHQ9RmFsc2UpLmZpdChYLCB5KS5jb2VmXw=='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import json, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(0)

## 1. MLE for a log-normal: the outage durations
Probability 2 found outage hours roughly log-normal: their logs are roughly normal. The MLE of a normal is the
sample mean and the variance dividing by n. Set `mu_hat` and `sigma_hat` for the natural log of `hours`, and
check them against SciPy's fitter `stats.norm.fit`.

In [ ]:
DATA = "https://raw.githubusercontent.com/ikruchynskyi/ihor.sh/main/ai/data/"
hours = np.array([r[0] for r in json.load(urllib.request.urlopen(DATA + "outages.json"))["rows"]])
mu_hat = sigma_hat = None
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...
print(mu_hat, sigma_hat, " typical outage:", np.exp(mu_hat) if mu_hat is not None else None, "hours")

In [ ]:
check(1, lambda: np.allclose([mu_hat, sigma_hat], stats.norm.fit(np.log(hours))), 
      "Sample mean and standard deviation of log(hours) should match stats.norm.fit.")
check(1, lambda: abs(np.exp(mu_hat) - 1.85) < 0.1, 
      "The geometric mean / median outage exp(mu_hat) should be ~1.85 hours.")

## 2. MLE by gradient descent
The same fit without a formula: minimize the negative log-likelihood of a normal,
$\sum_i \frac{(x_i - \mu)^2}{2\sigma^2} + n \ln \sigma$, over μ and $s = \ln\sigma$ (so σ stays positive) with
PyTorch and `torch.optim.Adam(lr=0.05)` for 1000 steps. Set `mu_gd` and `sigma_gd`.

In [ ]:
import torch
x = torch.tensor(np.log(hours))
mu_gd = sigma_gd = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
print(mu_gd, sigma_gd)

In [ ]:
check(2, lambda: abs(mu_gd - mu_hat) < 1e-3 and abs(sigma_gd - sigma_hat) < 1e-3, 
      "Gradient descent minimizing negative log-likelihood should converge to closed-form MLE parameters.")

## 3. The central limit theorem
Make 100,000 sums of 30 exponential waits each (`rng.exponential(4, (100_000, 30)).sum(axis=1)`). A single wait
is very skewed (skewness 2). Set `skew_sum` using `skew` (`stats.skew`) and plot the histogram: the sum is nearly a bell.

In [ ]:
skew_sum = sums = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
if sums is not None and len(sums):
    plt.hist(sums, bins=80, density=True); plt.title("sums of 30 waits"); plt.show()
print(skew_sum)

In [ ]:
check(3, lambda: 0.2 < skew_sum < 0.5, 
      "Skewness of sums of 30 variables should fall to ~2/sqrt(30) ≈ 0.37 (approaching a normal distribution).")

## 4. A Beta prior meets a coin
A coin of unknown bias. Prior Beta(2, 2). You see 7 heads in 10 flips. Set `a_post`, `b_post` (the posterior's
parameters), `p_map` (its peak), and `interval`: the central 95% credible interval with `beta` quantiles via `ppf` (`stats.beta.ppf` at 0.025 and
0.975).

In [ ]:
a_post = b_post = p_map = interval = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print(a_post, b_post, p_map, interval)

In [ ]:
check(4, lambda: (a_post, b_post) == (9, 5) and np.isclose(p_map, 2 / 3), 
      "Posterior parameters should be Beta(9, 5) with mode (9 - 1) / (9 + 5 - 2) = 8/12 = 2/3.")
check(4, lambda: interval[0] < 0.5 < interval[1], 
      "The 95% credible interval should still contain 0.5 (fair coin remains plausible with only 10 flips).")

## 5. Ridge is MAP
For a linear model with Gaussian noise σ and a Gaussian prior on the weights with spread τ, the MAP weights solve
$(X^\top X + \lambda I) w = X^\top y$ with $\lambda = \sigma^2/\tau^2$. On the standardized diabetes data
(no intercept: center y), set `w_map` from that formula with λ = 50, and `w_ridge` from
`Ridge(alpha=50, fit_intercept=False)`.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge
d = load_diabetes()
X = (d.data - d.data.mean(0)) / d.data.std(0); y = d.target - d.target.mean()
w_map = w_ridge = None
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
print(np.round(w_map, 2) if w_map is not None else None)

In [ ]:
check(5, lambda: np.allclose(w_map, w_ridge), 
      "The analytical MAP formula with Gaussian prior must exactly match sklearn Ridge(alpha=50).")
check(5, lambda: np.linalg.norm(w_map) < np.linalg.norm(np.linalg.lstsq(X, y, rcond=None)[0]), 
      "Ridge/MAP weights should have a smaller L2 norm than unregularized ordinary least squares.")

## Next
Back to the path: https://ihor.sh/ai/ · next: deep networks, layer by layer.